# SAE Evaluation — Colour Spurious Feature Analysis

This notebook mirrors the zebra-stripe tester exactly in its loading / SAE / LLaVA
infrastructure, and replaces the **experiment section** with colour-specific probes.

Dataset layout expected (upload as a Kaggle dataset):
```
paired_truck_dataset/
    Red/      ← original fire-truck images
    White/    ← desaturated versions (same filenames)
    Yellow/   ← hue-shifted to yellow
    Blue/     ← hue-shifted to blue
```

### Experiments we run
| # | Pair | What it isolates |
|---|------|------------------|
| A | Red vs White | chromatic-red signal (removes all hue) |
| B | Red vs Blue  | red vs cold-hue contrast |
| C | Red vs Yellow| red vs warm-hue contrast |
| D | Yellow vs Blue | warm vs cool hue (no red at all) |
| E | White vs {Red,Yellow,Blue} average | colourlessness / achromatic feature |
| F | Red vs White (selectivity sweep) | per-feature selectivity ranking |

---
### Prerequisites
| Kaggle input | What goes here |
|---|---|
| `clip-v14-activations` | `.pt` activation chunks from Step 1 (val split) |
| **Your uploaded SAE model** | `ae.pt` + `config.json` |
| `stable-imagenet1k` | Full ImageNet images |
| **paired_truck_dataset** | Generated by `dataset-maker.ipynb` |

## 0 · Setup  *(unchanged from zebra tester)*

In [ ]:
# ── Clone required repos ──────────────────────────────────────────────────────
!git clone https://github.com/saprmarks/dictionary_learning.git /kaggle/working/dictionary_learning 2>/dev/null || echo 'Already cloned'
!git clone https://github.com/ExplainableML/sae-for-vlm.git     /kaggle/working/sae-for-vlm      2>/dev/null || echo 'Already cloned'

!pip install -q -r /kaggle/working/sae-for-vlm/requirements.txt 2>/dev/null || true
!pip install -q transformers accelerate bitsandbytes Pillow tqdm matplotlib

import sys
sys.path.insert(0, "/kaggle/working/dictionary_learning")
sys.path.insert(0, "/kaggle/working/sae-for-vlm")
print("Setup complete")

import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
import os, glob, json, gc, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── PATHS ─────────────────────────────────────────────────────────────────────
SAE_MODEL_DIR = "/kaggle/input/models/akgiiith/sae-clip-vit-l14-layer-22-batchtopk/pytorch/default/1"
SAE_CKPT      = os.path.join(SAE_MODEL_DIR, "ae.pt")
SAE_CONFIG    = os.path.join(SAE_MODEL_DIR, "config.json")

VAL_ACT_DIR   = "/kaggle/input/datasets/akgiiith/clip-v14-activations/val"
IMAGENET_ROOT = "/kaggle/input/datasets/vitaliykinakh/stable-imagenet1k/imagenet1k"

# ← UPDATE to wherever you uploaded the paired_truck_dataset
COLOUR_ROOT   = "/kaggle/input/datasets/<YOUR_USERNAME>/paired-truck-dataset/paired_truck_dataset"

ACTIVATION_DIM = 1024
DICT_SIZE      = 4096
K              = 20
TARGET_LAYER   = 22

print(f"SAE checkpoint : {SAE_CKPT}")
print(f"Val activations: {VAL_ACT_DIR}")
print(f"Colour dataset : {COLOUR_ROOT}")

---
## 1 · Load SAE from Checkpoint  *(unchanged)*

In [ ]:
with open(SAE_CONFIG) as f:
    cfg = json.load(f)
print("Config from checkpoint:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

ACTIVATION_DIM = cfg.get("activation_dim", ACTIVATION_DIM)
DICT_SIZE      = cfg.get("dict_size",      DICT_SIZE)
K              = cfg.get("k",              K)

In [ ]:
def load_sae_from_checkpoint(ckpt_path, cfg):
    try:
        obj = torch.load(ckpt_path, map_location=DEVICE)
        if isinstance(obj, nn.Module):
            print("Loaded SAE as full module object")
            return obj.to(DEVICE)
    except Exception:
        pass

    from dictionary import BatchTopKSAE
    act_dim  = cfg["trainer"]["activation_dim"]
    dict_sz  = cfg["trainer"]["dict_size"]
    k_val    = cfg["trainer"]["k"]
    sae = BatchTopKSAE(activation_dim=act_dim, dict_size=dict_sz, k=k_val)
    sd = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(sd, dict):
        sae.load_state_dict(sd)
    sae = sae.to(DEVICE)
    print(f"Loaded SAE: dim={act_dim}, dict={dict_sz}, k={k_val}")
    return sae

ae = load_sae_from_checkpoint(SAE_CKPT, cfg)
ae.eval()
print("SAE ready.")

---
## 2 · Sanity Metrics on Val Set  *(unchanged)*

In [ ]:
val_chunks = sorted(glob.glob(os.path.join(VAL_ACT_DIR, "*.pt")))
print(f"Val chunks found: {len(val_chunks)}")

all_acts, all_recons = [], []
with torch.no_grad():
    for chunk_path in tqdm(val_chunks[:10], desc="Evaluating"):
        acts = torch.load(chunk_path, map_location=DEVICE).float()
        out  = ae(acts)
        recon = out[0] if isinstance(out, (tuple, list)) else out
        all_acts.append(acts.cpu())
        all_recons.append(recon.cpu())

acts_cat   = torch.cat(all_acts)
recon_cat  = torch.cat(all_recons)

mse      = F.mse_loss(recon_cat, acts_cat).item()
ss_res   = ((acts_cat - recon_cat) ** 2).sum().item()
ss_tot   = ((acts_cat - acts_cat.mean(0)) ** 2).sum().item()
r2       = 1 - ss_res / ss_tot

with torch.no_grad():
    l0_vals = []
    for chunk_path in val_chunks[:10]:
        acts = torch.load(chunk_path, map_location=DEVICE).float()
        out  = ae(acts)
        feat = out[1] if isinstance(out, (tuple, list)) and len(out) > 1 else ae.encode(acts)
        l0_vals.append((feat != 0).float().sum(-1).mean().item())
l0 = np.mean(l0_vals)

all_feat_flat = []
with torch.no_grad():
    for chunk_path in val_chunks[:10]:
        acts = torch.load(chunk_path, map_location=DEVICE).float()
        out  = ae(acts)
        feat = out[1] if isinstance(out, (tuple, list)) and len(out) > 1 else ae.encode(acts)
        all_feat_flat.append(feat.cpu())
feat_cat  = torch.cat(all_feat_flat)
active    = (feat_cat > 0).float().mean(0)
pct_dead  = (active == 0).float().mean().item() * 100

print(f"R²        : {r2:.4f}")
print(f"MSE       : {mse:.6f}")
print(f"L0 (mean) : {l0:.2f}  (target={K})")
print(f"Dead feat : {pct_dead:.1f}%")

---
## 3 · CLIP Image Encoder  *(unchanged)*

In [ ]:
import clip

clip_model, clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
print("CLIP ViT-L/14 ready")

@torch.no_grad()
def get_clip_activation(img_path_or_pil, layer_idx=TARGET_LAYER):
    """Returns the [CLS] activation at layer_idx for one image."""
    if isinstance(img_path_or_pil, str):
        img = Image.open(img_path_or_pil).convert("RGB")
    else:
        img = img_path_or_pil
    x = clip_preprocess(img).unsqueeze(0).to(DEVICE)

    activations = {}
    def hook_fn(module, inp, out):
        activations["feat"] = out[:, 0, :]   # CLS token

    handle = clip_model.visual.transformer.resblocks[layer_idx].register_forward_hook(hook_fn)
    try:
        clip_model.encode_image(x)
    finally:
        handle.remove()
    return activations["feat"].squeeze(0).float()


@torch.no_grad()
def get_sae_features(img_path_or_pil):
    """Returns sparse SAE feature vector for one image."""
    act = get_clip_activation(img_path_or_pil).unsqueeze(0)
    out = ae(act)
    feat = out[1] if isinstance(out, (tuple, list)) and len(out) > 1 else ae.encode(act)
    return feat.squeeze(0)

print("Helper functions defined.")

---
## 4 · Build Colour Activation Maps

For each colour folder we compute the **mean SAE feature vector** over all images.
All four folders contain the **same images** with only the truck colour changed,
so differences in mean activations are caused entirely by colour.

We also build per-image paired tensors for later statistical tests.

In [ ]:
COLOUR_FOLDERS = ["Red", "White", "Yellow", "Blue"]

# Confirm all folders exist and list common filenames
folder_files = {}
for c in COLOUR_FOLDERS:
    path = os.path.join(COLOUR_ROOT, c)
    files = sorted([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    folder_files[c] = files
    print(f"  {c}: {len(files)} images")

# Only keep filenames present in ALL four folders
common_files = sorted(set(folder_files["Red"]) & set(folder_files["White"]) \
                      & set(folder_files["Yellow"]) & set(folder_files["Blue"]))
print(f"\nCommon (paired) images: {len(common_files)}")

In [ ]:
# ── Compute per-image SAE features for every colour ──────────────────────────
# Result shape: {colour: (N, DICT_SIZE)}

colour_feats = {c: [] for c in COLOUR_FOLDERS}

for fname in tqdm(common_files, desc="Encoding images"):
    for c in COLOUR_FOLDERS:
        fpath = os.path.join(COLOUR_ROOT, c, fname)
        feat  = get_sae_features(fpath).cpu()
        colour_feats[c].append(feat)

for c in COLOUR_FOLDERS:
    colour_feats[c] = torch.stack(colour_feats[c])   # (N, 4096)
    print(f"  {c} tensor: {colour_feats[c].shape}")

# Mean vectors per colour
colour_mean = {c: colour_feats[c].mean(0) for c in COLOUR_FOLDERS}
print("Mean feature vectors computed.")

---
## 5 · Experiment A — Red vs White  (chromatic-red signal)

White trucks have been desaturated: only the *hue* information is removed.
Features that activate more on Red than White encode **red chromatic content**.

In [ ]:
# ── Selectivity: mean(Red) / (mean(White) + eps) ─────────────────────────────
eps = 1e-8
sel_red_vs_white = colour_mean["Red"] / (colour_mean["White"] + eps)

top_red_feats = sel_red_vs_white.argsort(descending=True)[:20].tolist()
print("Top-20 RED-selective features (Red vs White):")
print("-" * 55)
print(f"{'Rank':>4}  {'Feat#':>6}  {'Red mean':>9}  {'White mean':>10}  {'Sel ratio':>9}")
print("-" * 55)
for rank, fid in enumerate(top_red_feats, 1):
    rm  = colour_mean["Red"][fid].item()
    wm  = colour_mean["White"][fid].item()
    sel = sel_red_vs_white[fid].item()
    print(f"  {rank:2d}    #{fid:4d}    {rm:9.4f}    {wm:10.4f}    {sel:9.1f}×")

SPOTLIGHT_RED = top_red_feats[0]
print(f"\n★  Spotlight feature (Red vs White): #{SPOTLIGHT_RED}")

In [ ]:
# ── Distribution plot: activation of top red feature across all 4 colours ─────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: violin plot
data_by_colour = [colour_feats[c][:, SPOTLIGHT_RED].numpy() for c in COLOUR_FOLDERS]
colour_hex     = ["#E74C3C", "#ECF0F1", "#F1C40F", "#3498DB"]

parts = axes[0].violinplot(data_by_colour, showmedians=True)
for pc, col in zip(parts["bodies"], colour_hex):
    pc.set_facecolor(col)
    pc.set_edgecolor("black")
    pc.set_alpha(0.85)
axes[0].set_xticks(range(1, 5))
axes[0].set_xticklabels(COLOUR_FOLDERS)
axes[0].set_title(f"Feature #{SPOTLIGHT_RED} activation by truck colour")
axes[0].set_ylabel("SAE activation")
axes[0].set_xlabel("Truck colour")

# Right: bar of mean ± std
means  = [colour_feats[c][:, SPOTLIGHT_RED].mean().item() for c in COLOUR_FOLDERS]
stds   = [colour_feats[c][:, SPOTLIGHT_RED].std().item()  for c in COLOUR_FOLDERS]
axes[1].bar(COLOUR_FOLDERS, means, yerr=stds, color=colour_hex, edgecolor="black", capsize=6)
axes[1].set_title(f"Mean (±σ) of Feature #{SPOTLIGHT_RED}")
axes[1].set_ylabel("Mean SAE activation")

plt.tight_layout()
plt.savefig("/kaggle/working/exp_A_red_vs_white.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_A_red_vs_white.png")

In [ ]:
# ── Paired t-test: is the Red vs White difference statistically significant? ──
from scipy import stats

red_acts   = colour_feats["Red"][:,   SPOTLIGHT_RED].numpy()
white_acts = colour_feats["White"][:, SPOTLIGHT_RED].numpy()

t_stat, p_val = stats.ttest_rel(red_acts, white_acts)
print(f"Paired t-test (Red vs White) for feature #{SPOTLIGHT_RED}")
print(f"  t = {t_stat:.3f},  p = {p_val:.2e}")
print(f"  {'✅ Significant (p<0.01)' if p_val < 0.01 else '⚠️ Not significant'}")

---
## 6 · Experiment B — Red vs Blue  (red vs cold-hue)

Both are fully saturated colours; difference encodes **hue direction** (warm vs cool).

In [ ]:
sel_red_vs_blue = colour_mean["Red"] / (colour_mean["Blue"] + eps)
sel_blue_vs_red = colour_mean["Blue"] / (colour_mean["Red"] + eps)

top_red_blue_feats  = sel_red_vs_blue.argsort(descending=True)[:10].tolist()
top_blue_red_feats  = sel_blue_vs_red.argsort(descending=True)[:10].tolist()

print("Top-10 RED-selective features (Red vs Blue):")
for rank, fid in enumerate(top_red_blue_feats, 1):
    rm = colour_mean["Red"][fid].item()
    bm = colour_mean["Blue"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  red={rm:.4f}  blue={bm:.4f}  ratio={rm/(bm+eps):.1f}×")

print("\nTop-10 BLUE-selective features (Blue vs Red):")
for rank, fid in enumerate(top_blue_red_feats, 1):
    rm = colour_mean["Red"][fid].item()
    bm = colour_mean["Blue"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  blue={bm:.4f}  red={rm:.4f}  ratio={bm/(rm+eps):.1f}×")

SPOTLIGHT_BLUE = top_blue_red_feats[0]
print(f"\n★  Blue-selective spotlight: #{SPOTLIGHT_BLUE}")

In [ ]:
# ── Scatter: per-image Red activation vs Blue activation ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (feat_id, label) in zip(axes,
        [(SPOTLIGHT_RED, f"Red-selective #{SPOTLIGHT_RED}"),
         (SPOTLIGHT_BLUE, f"Blue-selective #{SPOTLIGHT_BLUE}")]):
    rx = colour_feats["Red"][:,  feat_id].numpy()
    bx = colour_feats["Blue"][:, feat_id].numpy()
    ax.scatter(rx, bx, alpha=0.6, edgecolors="k", linewidths=0.5, s=50)
    lim = max(rx.max(), bx.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1, label="y=x")
    ax.set_xlabel("Red truck activation")
    ax.set_ylabel("Blue truck activation")
    ax.set_title(label)
    ax.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/exp_B_red_vs_blue.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_B_red_vs_blue.png")

---
## 7 · Experiment C — Red vs Yellow  (red vs warm-hue)

Both are warm colours; this isolates features that specifically encode **redness**
over generic warm-hue energy.

In [ ]:
sel_red_vs_yellow    = colour_mean["Red"]    / (colour_mean["Yellow"] + eps)
sel_yellow_vs_red    = colour_mean["Yellow"] / (colour_mean["Red"]    + eps)

top_red_yellow       = sel_red_vs_yellow.argsort(descending=True)[:10].tolist()
top_yellow_red       = sel_yellow_vs_red.argsort(descending=True)[:10].tolist()

print("Top-10 RED-specific features (Red vs Yellow — both warm):")
for rank, fid in enumerate(top_red_yellow, 1):
    rm = colour_mean["Red"][fid].item()
    ym = colour_mean["Yellow"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  red={rm:.4f}  yellow={ym:.4f}  ratio={rm/(ym+eps):.1f}×")

print("\nTop-10 YELLOW-specific features (Yellow vs Red):")
for rank, fid in enumerate(top_yellow_red, 1):
    rm = colour_mean["Red"][fid].item()
    ym = colour_mean["Yellow"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  yellow={ym:.4f}  red={rm:.4f}  ratio={ym/(rm+eps):.1f}×")

SPOTLIGHT_YELLOW = top_yellow_red[0]
print(f"\n★  Yellow-selective spotlight: #{SPOTLIGHT_YELLOW}")

In [ ]:
# ── Heatmap: pairwise mean activation comparison across all 4 colours ─────────
# Row = colour A, Col = colour B: value is mean feature activation for that colour
# Use only the union of top features across all pair-wise comparisons

top_feats_all = sorted(set(
    top_red_feats[:8] + top_red_blue_feats[:5] + top_red_yellow[:5] +
    top_yellow_red[:5] + top_blue_red_feats[:5]
))

heatmap_data = np.array([
    [colour_mean[c][fid].item() for fid in top_feats_all]
    for c in COLOUR_FOLDERS
])  # shape: (4, len(top_feats_all))

fig, ax = plt.subplots(figsize=(max(12, len(top_feats_all) * 0.7), 3.5))
im = ax.imshow(heatmap_data, aspect="auto", cmap="RdYlBu_r")
ax.set_yticks(range(4))
ax.set_yticklabels(COLOUR_FOLDERS)
ax.set_xticks(range(len(top_feats_all)))
ax.set_xticklabels([f"#{f}" for f in top_feats_all], rotation=70, fontsize=8)
ax.set_title("Mean SAE activation — top colour-discriminating features")
plt.colorbar(im, ax=ax, label="Mean activation")
plt.tight_layout()
plt.savefig("/kaggle/working/exp_C_colour_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_C_colour_heatmap.png")

---
## 8 · Experiment D — Yellow vs Blue  (warm vs cool, no red)

Neither colour is red, so any discovered features are truly encoding **hue polarity**
independent of the red training signal.

In [ ]:
sel_yel_vs_blue = colour_mean["Yellow"] / (colour_mean["Blue"] + eps)
sel_blu_vs_yel  = colour_mean["Blue"]   / (colour_mean["Yellow"] + eps)

top_yel_blue = sel_yel_vs_blue.argsort(descending=True)[:10].tolist()
top_blu_yel  = sel_blu_vs_yel.argsort(descending=True)[:10].tolist()

print("Top-10 YELLOW-selective (vs Blue) — pure warm-hue features:")
for rank, fid in enumerate(top_yel_blue, 1):
    ym = colour_mean["Yellow"][fid].item()
    bm = colour_mean["Blue"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  yellow={ym:.4f}  blue={bm:.4f}  ratio={ym/(bm+eps):.1f}×")

print("\nTop-10 BLUE-selective (vs Yellow) — pure cool-hue features:")
for rank, fid in enumerate(top_blu_yel, 1):
    ym = colour_mean["Yellow"][fid].item()
    bm = colour_mean["Blue"][fid].item()
    print(f"  {rank:2d}. #{fid:4d}  blue={bm:.4f}  yellow={ym:.4f}  ratio={bm/(ym+eps):.1f}×")

# Check overlap with red-selective features — tells us if SAE reuses same neurons
red_set = set(top_red_feats[:20])
yel_set = set(top_yel_blue)
blu_set = set(top_blu_yel)
print(f"\nOverlap (Red-selective ∩ Yellow-warm): {red_set & yel_set}")
print(f"Overlap (Red-selective ∩ Blue-cool)  : {red_set & blu_set}")

In [ ]:
# ── Activation trajectory per image across the 4 colours ─────────────────────
# Pick 5 random images and plot their feature activations across all colours

N_TRACE = 5
trace_indices = np.random.choice(len(common_files), N_TRACE, replace=False)

fig, axes = plt.subplots(1, N_TRACE, figsize=(4 * N_TRACE, 4), sharey=True)
colour_plot_cols = ["#E74C3C", "#BDC3C7", "#F1C40F", "#2980B9"]

for ax_idx, img_idx in enumerate(trace_indices):
    feat_vals = [colour_feats[c][img_idx, SPOTLIGHT_RED].item() for c in COLOUR_FOLDERS]
    ax = axes[ax_idx]
    ax.bar(COLOUR_FOLDERS, feat_vals, color=colour_plot_cols, edgecolor="black")
    ax.set_title(f"Img {img_idx}\n{common_files[img_idx][:15]}", fontsize=8)
    ax.set_xlabel("Colour")
    if ax_idx == 0:
        ax.set_ylabel(f"Feature #{SPOTLIGHT_RED} activation")

plt.suptitle(f"Per-image activation of feature #{SPOTLIGHT_RED} across truck colours", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/exp_D_per_image_trace.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_D_per_image_trace.png")

---
## 9 · Experiment E — Colourlessness (White vs Chromatic Average)

We compute a *chromatic average* = mean(Red, Yellow, Blue) to represent the
"any-colour" signal.  Features that activate **more on White** than this average
are **achromatic / colourlessness** features.

In [ ]:
chromatic_mean = (colour_mean["Red"] + colour_mean["Yellow"] + colour_mean["Blue"]) / 3.0

sel_white_vs_chroma = colour_mean["White"] / (chromatic_mean + eps)
sel_chroma_vs_white = chromatic_mean        / (colour_mean["White"] + eps)

top_white_feats   = sel_white_vs_chroma.argsort(descending=True)[:15].tolist()
top_chroma_feats  = sel_chroma_vs_white.argsort(descending=True)[:15].tolist()

print("Top-15 ACHROMATIC (White-selective) features:")
print("-" * 70)
print(f"{'Rank':>4}  {'Feat#':>6}  {'White':>8}  {'Chroma avg':>11}  {'Sel':>8}")
print("-" * 70)
for rank, fid in enumerate(top_white_feats, 1):
    wm = colour_mean["White"][fid].item()
    cm = chromatic_mean[fid].item()
    print(f"  {rank:2d}    #{fid:4d}    {wm:8.4f}    {cm:11.4f}    {wm/(cm+eps):8.1f}×")

SPOTLIGHT_WHITE = top_white_feats[0]
print(f"\n★  Achromatic spotlight feature: #{SPOTLIGHT_WHITE}")

print("\nTop-15 CHROMATIC (any colour vs White) features:")
for rank, fid in enumerate(top_chroma_feats, 1):
    wm = colour_mean["White"][fid].item()
    cm = chromatic_mean[fid].item()
    print(f"  {rank:2d}. #{fid:4d}  chroma_avg={cm:.4f}  white={wm:.4f}  ratio={cm/(wm+eps):.1f}×")

In [ ]:
# ── Visualise top-N images that maximally activate the achromatic feature ─────
N_VIZ = 8
white_acts_all = colour_feats["White"][:, SPOTLIGHT_WHITE].numpy()
top_img_indices = white_acts_all.argsort()[::-1][:N_VIZ]

fig, axes = plt.subplots(2, N_VIZ, figsize=(2.5 * N_VIZ, 6))
for col, img_idx in enumerate(top_img_indices):
    fname = common_files[img_idx]
    for row, c in enumerate(["White", "Red"]):
        path = os.path.join(COLOUR_ROOT, c, fname)
        img  = Image.open(path).convert("RGB")
        ax   = axes[row, col]
        ax.imshow(img)
        act_val = colour_feats[c][img_idx, SPOTLIGHT_WHITE].item()
        ax.set_title(f"{c}\n{act_val:.3f}", fontsize=7)
        ax.axis("off")

axes[0, 0].set_ylabel("White version", fontsize=9)
axes[1, 0].set_ylabel("Red version",   fontsize=9)
fig.suptitle(f"Top-{N_VIZ} images for achromatic feature #{SPOTLIGHT_WHITE}\n(top row=White, bottom=Red)",
             fontsize=11)
plt.tight_layout()
plt.savefig("/kaggle/working/exp_E_achromatic_gallery.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_E_achromatic_gallery.png")

---
## 10 · Experiment F — Selectivity Sweep + Ranked Feature Gallery

For the **red-vs-white** spotlight feature we show the top- and bottom-activating
images (both colours side by side) to visually confirm the feature is colour-specific
and not confounded by truck shape or background.

In [ ]:
# ── Selectivity histogram across all 4096 features ───────────────────────────
all_sel_rw = (colour_mean["Red"] / (colour_mean["White"] + eps)).numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_sel_rw[all_sel_rw < 50], bins=100, color="#E74C3C", edgecolor="white", alpha=0.8)
ax.axvline(all_sel_rw[SPOTLIGHT_RED], color="black", linestyle="--",
           label=f"Spotlight #{SPOTLIGHT_RED} ({all_sel_rw[SPOTLIGHT_RED]:.1f}×)")
ax.set_xlabel("Red / White selectivity ratio")
ax.set_ylabel("Number of SAE features")
ax.set_title("Distribution of Red-vs-White selectivity across all 4096 features")
ax.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/exp_F_selectivity_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_F_selectivity_hist.png")

In [ ]:
# ── Gallery: top-8 vs bottom-8 activating images for spotlight red feature ────
red_acts_all = colour_feats["Red"][:, SPOTLIGHT_RED].numpy()
top8_idx     = red_acts_all.argsort()[::-1][:8]
bot8_idx     = red_acts_all.argsort()[:8]

def show_paired_gallery(indices, title, filename):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.5 * len(indices), 6))
    for col, img_idx in enumerate(indices):
        fname = common_files[img_idx]
        for row, c in enumerate(["Red", "White"]):
            path = os.path.join(COLOUR_ROOT, c, fname)
            img  = Image.open(path).convert("RGB")
            ax   = axes[row, col]
            ax.imshow(img)
            act_val = colour_feats[c][img_idx, SPOTLIGHT_RED].item()
            ax.set_title(f"{c}: {act_val:.3f}", fontsize=7)
            ax.axis("off")
    axes[0, 0].set_ylabel("Red",   fontsize=9)
    axes[1, 0].set_ylabel("White", fontsize=9)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/{filename}", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {filename}")

show_paired_gallery(top8_idx, f"TOP-8 activating images — Red feature #{SPOTLIGHT_RED}",
                    "exp_F_top8_red.png")
show_paired_gallery(bot8_idx, f"BOTTOM-8 activating images — Red feature #{SPOTLIGHT_RED}",
                    "exp_F_bot8_red.png")

---
## 11 · Experiment G — Multi-Feature Colour Fingerprint

For each colour we compute its **top-K active features** and measure
overlap between colours.  A high overlap between Red and Yellow (warm pair)
but low overlap with Blue / White confirms the SAE has learned hue-specific features.

In [ ]:
TOP_K_FINGERPRINT = 50

colour_topk = {
    c: set(colour_mean[c].argsort(descending=True)[:TOP_K_FINGERPRINT].tolist())
    for c in COLOUR_FOLDERS
}

print(f"Top-{TOP_K_FINGERPRINT} feature overlap between colour pairs:")
print("-" * 45)
pairs = [("Red","White"),("Red","Yellow"),("Red","Blue"),("Yellow","Blue"),("White","Blue"),("White","Yellow")]
for ca, cb in pairs:
    overlap = len(colour_topk[ca] & colour_topk[cb])
    print(f"  {ca:6s} ∩ {cb:6s}: {overlap:2d} / {TOP_K_FINGERPRINT}  "
          f"({'high' if overlap > 25 else 'medium' if overlap > 12 else 'low'})")

# Jaccard similarity matrix
print("\nJaccard similarity matrix:")
print(f"{'':8s}", end="")
for c in COLOUR_FOLDERS:
    print(f"{c:8s}", end="")
print()
for ca in COLOUR_FOLDERS:
    print(f"{ca:8s}", end="")
    for cb in COLOUR_FOLDERS:
        inter = len(colour_topk[ca] & colour_topk[cb])
        union = len(colour_topk[ca] | colour_topk[cb])
        jac   = inter / union
        print(f"{jac:8.3f}", end="")
    print()

In [ ]:
# ── Jaccard heatmap ───────────────────────────────────────────────────────────
jac_matrix = np.zeros((4, 4))
for i, ca in enumerate(COLOUR_FOLDERS):
    for j, cb in enumerate(COLOUR_FOLDERS):
        inter = len(colour_topk[ca] & colour_topk[cb])
        union = len(colour_topk[ca] | colour_topk[cb])
        jac_matrix[i, j] = inter / union

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(jac_matrix, cmap="Greens", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Jaccard similarity")
ax.set_xticks(range(4)); ax.set_xticklabels(COLOUR_FOLDERS)
ax.set_yticks(range(4)); ax.set_yticklabels(COLOUR_FOLDERS)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{jac_matrix[i,j]:.2f}", ha="center", va="center", fontsize=11)
ax.set_title(f"Feature-set Jaccard similarity (top-{TOP_K_FINGERPRINT} per colour)")
plt.tight_layout()
plt.savefig("/kaggle/working/exp_G_jaccard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp_G_jaccard.png")

---
## 12 · Load LLaVA-NeXT  *(unchanged from zebra tester)*

In [ ]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
import torch

LLAVA_MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"

llava_proc  = LlavaNextProcessor.from_pretrained(LLAVA_MODEL_ID)
llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    torch_dtype=torch.float16,
    load_in_4bit=True,
    device_map="auto"
)
llava_model.eval()
print("LLaVA-NeXT ready")

def run_inference(model, processor, image, question, max_new=80):
    conversation = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": question}
    ]}]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    full    = processor.decode(out[0], skip_special_tokens=True)
    answer  = full.split("assistant")[-1].strip() if "assistant" in full else full
    return answer

print("run_inference helper ready.")

In [ ]:
def get_vision_layer(model, layer_idx):
    """Returns the transformer block at layer_idx in the CLIP vision tower."""
    vt = model.model.vision_tower
    return vt.vision_model.encoder.layers[layer_idx]

def make_clamp_hook(sae, feature_ids, device, clamp_value=0.0):
    """Returns a forward hook that zeroes out the specified SAE feature directions."""
    def hook(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        with torch.no_grad():
            for fid in feature_ids:
                direction = sae.W_dec[fid].to(device)
                direction = direction / (direction.norm() + 1e-8)
                proj = (hidden * direction).sum(-1, keepdim=True)
                hidden = hidden - proj * direction
        return (hidden,) + out[1:] if isinstance(out, tuple) else hidden
    return hook

print("VLM hook helpers defined.")

---
## 13 · Moment of Truth — Colour Feature Steering

We run four steering experiments:
1. **Suppress Red features** on Red trucks → does model stop saying "red"?
2. **Suppress Red features** on Blue trucks → collateral damage check
3. **Suppress Achromatic features** on White truck → does colour description change?
4. **Cross-colour steering** — inject Red direction into White truck → does model hallucinate red?

In [ ]:
# Pick 5 test images (first 5 common files)
N_TEST = 5
test_files = common_files[:N_TEST]

COLOUR_QUESTION = "What color is the truck in this image?"

# ── Steering Experiment 1: Suppress Red features on Red trucks ────────────────
print("Steering Experiment 1: Suppress top Red features on RED trucks")
print("=" * 65)

for fname in test_files:
    red_path  = os.path.join(COLOUR_ROOT, "Red", fname)
    red_img   = Image.open(red_path).convert("RGB")

    r_base  = run_inference(llava_model, llava_proc, red_img, COLOUR_QUESTION)

    layer  = get_vision_layer(llava_model, TARGET_LAYER)
    hook   = make_clamp_hook(ae, top_red_feats[:10], DEVICE)
    handle = layer.register_forward_hook(hook)
    try:
        r_steer = run_inference(llava_model, llava_proc, red_img, COLOUR_QUESTION)
    finally:
        handle.remove()

    red_lost = "red" not in r_steer.lower() and "red" in r_base.lower()
    sym = "✅" if red_lost else "⚠️"
    print(f"  {fname[:30]:30s}  base: '{r_base[:45]}'")
    print(f"  {' '*30}  steer: '{r_steer[:45]}'  {sym}")
    print()

In [ ]:
# ── Steering Experiment 2: Collateral damage — suppress Red on BLUE trucks ────
print("Steering Experiment 2: Suppress Red features on BLUE trucks (collateral check)")
print("=" * 70)

correct_base, correct_steered = 0, 0

for fname in test_files:
    blue_path = os.path.join(COLOUR_ROOT, "Blue", fname)
    blue_img  = Image.open(blue_path).convert("RGB")

    r_base = run_inference(llava_model, llava_proc, blue_img, COLOUR_QUESTION)
    base_ok = "blue" in r_base.lower()
    correct_base += int(base_ok)

    layer  = get_vision_layer(llava_model, TARGET_LAYER)
    hook   = make_clamp_hook(ae, top_red_feats[:10], DEVICE)
    handle = layer.register_forward_hook(hook)
    try:
        r_steer = run_inference(llava_model, llava_proc, blue_img, COLOUR_QUESTION)
    finally:
        handle.remove()
    steer_ok = "blue" in r_steer.lower()
    correct_steered += int(steer_ok)

    sym = "✅" if steer_ok else "❌"
    print(f"  {fname[:28]:28s}  base: '{r_base[:40]}'  steer: '{r_steer[:40]}'  {sym}")

print(f"\n  Blue accuracy  base  : {correct_base}/{N_TEST}")
print(f"  Blue accuracy steered: {correct_steered}/{N_TEST}")
print(f"  {'✅ No collateral damage' if correct_steered >= correct_base - 1 else '⚠️ Some collateral damage'}")

In [ ]:
# ── Steering Experiment 3: Suppress Achromatic features on White trucks ───────
print("Steering Experiment 3: Suppress Achromatic (White) features on WHITE trucks")
print("=" * 70)

for fname in test_files:
    white_path = os.path.join(COLOUR_ROOT, "White", fname)
    white_img  = Image.open(white_path).convert("RGB")

    r_base = run_inference(llava_model, llava_proc, white_img, COLOUR_QUESTION)

    layer  = get_vision_layer(llava_model, TARGET_LAYER)
    hook   = make_clamp_hook(ae, top_white_feats[:10], DEVICE)
    handle = layer.register_forward_hook(hook)
    try:
        r_steer = run_inference(llava_model, llava_proc, white_img, COLOUR_QUESTION)
    finally:
        handle.remove()

    print(f"  {fname[:30]:30s}  base: '{r_base[:45]}'")
    print(f"  {' '*30}  steer: '{r_steer[:45]}'")
    print()

In [ ]:
# ── Steering Experiment 4: Cross-colour — suppress Blue features on Blue trucks
# then ask if it now looks red. Tests directional feature specificity.
print("Steering Experiment 4: Suppress Blue features on BLUE trucks")
print("Does the model shift toward a different (warmer) colour description?")
print("=" * 65)

for fname in test_files:
    blue_path = os.path.join(COLOUR_ROOT, "Blue", fname)
    blue_img  = Image.open(blue_path).convert("RGB")

    r_base = run_inference(llava_model, llava_proc, blue_img, COLOUR_QUESTION)

    layer  = get_vision_layer(llava_model, TARGET_LAYER)
    hook   = make_clamp_hook(ae, top_blue_red_feats[:10], DEVICE)
    handle = layer.register_forward_hook(hook)
    try:
        r_steer = run_inference(llava_model, llava_proc, blue_img, COLOUR_QUESTION)
    finally:
        handle.remove()

    shifted = "blue" not in r_steer.lower()
    sym = "✅" if shifted else "—"
    print(f"  {fname[:28]:28s}  base: '{r_base[:40]}'")
    print(f"  {' '*28}  steer: '{r_steer[:40]}'  {sym}")
    print()

---
## 14 · Summary

In [ ]:
print("╔" + "═" * 62 + "╗")
print("║        SAE EVALUATION SUMMARY — COLOUR FEATURES            ║")
print("╠" + "═" * 62 + "╣")
print(f"║  Model        : BatchTopK SAE — layer {TARGET_LAYER} CLIP ViT-L/14      ║")
print(f"║  Dict size    : {DICT_SIZE} features  (exp ×{DICT_SIZE//ACTIVATION_DIM})                      ║")
print(f"║  K (sparsity) : {K}                                           ║")
print("╠" + "═" * 62 + "╣")
print("║  SANITY METRICS (val set)                                  ║")
print(f"║   R²           : {r2:.4f}                                     ║")
print(f"║   L0 (mean)    : {l0:.2f}   (target={K})                          ║")
print(f"║   MSE          : {mse:.6f}                                 ║")
print(f"║   Dead feat    : {pct_dead:.1f}%                                       ║")
print("╠" + "═" * 62 + "╣")
print("║  COLOUR FEATURES DISCOVERED                                ║")
print(f"║   Red spotlight       : #{SPOTLIGHT_RED:4d}                              ║")
print(f"║   Red sel (vs White)  : {sel_red_vs_white[SPOTLIGHT_RED]:.1f}×                              ║")
print(f"║   Blue spotlight      : #{SPOTLIGHT_BLUE:4d}                              ║")
print(f"║   Yellow spotlight    : #{SPOTLIGHT_YELLOW:4d}                              ║")
print(f"║   Achromatic spotl.   : #{SPOTLIGHT_WHITE:4d}                              ║")
print("╠" + "═" * 62 + "╣")
print("║  EXPERIMENTS RUN                                           ║")
print("║  A: Red vs White  (chromatic-red signal)                   ║")
print("║  B: Red vs Blue   (warm vs cool hue)                       ║")
print("║  C: Red vs Yellow (red vs warm-hue)                        ║")
print("║  D: Yellow vs Blue (hue polarity, no red)                  ║")
print("║  E: White vs Chromatic avg (achromatic features)           ║")
print("║  F: Selectivity sweep + ranked gallery                     ║")
print("║  G: Multi-colour feature-set Jaccard fingerprint           ║")
print("╠" + "═" * 62 + "╣")
print("║  MOMENT OF TRUTH (VLM colour steering)                     ║")
print("║  Exp 1: Suppress red  on Red trucks                        ║")
print("║  Exp 2: Suppress red  on Blue trucks (collateral check)    ║")
print("║  Exp 3: Suppress white on White trucks                     ║")
print("║  Exp 4: Suppress blue on Blue trucks (cross-colour shift)  ║")
print("╚" + "═" * 62 + "╝")